<a href="https://colab.research.google.com/github/schmitfe/Nawrot_CNS_Course/blob/main/Decoding_Motor_Cortex_Activity_Part_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decoding of movement direction from single unit activity in the monkey motor cortex

## Part II – Decoding neuronal population activity

**Martin Nawrot and Alexa Riehle**

Computational Systems Neuroscience and Animal Physiology, Institute for Zoology, Department of Biology, University of Cologne, D 50674 Cologne.

Institut de Neurosciences Cognitives de la Méditerranée, Centre National de la Recherche Scientifique (CNRS) and University of Aix-Marseille, F 13402 Marseille


## Synopsis

This course module extends the previous single-neuron analysis in Part I by introducing the time-resolved analysis of neuronal populations and basic machine-learning methods for classification. The students will represent directional information from monkey primary motor cortex during center-out arm reaching movements [1] using color-coded firing-rate displays and principal component analysis (PCA). Using Linear Discriminant Analysis (LDA) or a Naïve Bayes classifier, the students will construct a model that is trained and tested with cross-validation. This teaching module addresses master and graduate students from both experimental and theoretical backgrounds. The experiment and data are described in detail in [1].


## Supplemental Material

With this course module we provide additional data files for practical analysis. Data courtesy: *Alexa Riehle, Mediterranean Institute of Cognitive Neuroscience – Centre National de la Recherche Scientifique (CNRS) and University of Aix-Marseille, 13402 Marseille, France.*


##Introduction

In the early and mid 1980's Georgopoulus and coworkers showed that a large proportion of neurons in the primary motor cortex of the behaving monkey show a dependence of their firing probability on the direction of an arm movement [2]. This was studied in an experimental paradigm known as the 'center-out task' where the monkey has to perform arm movements from a central starting position (center) to one out of several targets which are arranged on an outer circle (out) around the central starting position. In the present exercises, you will analyze single unit recordings from the primary motor cortex (M1) of one monkey (monkey 1 in [1,7]), which performed a delayed center-out task. The experiments were carried out in the lab of Alexa Riehle. The task and the data are described in detail in [1]. Additional analyses on the trial-to-trial variability of the same data set is shown in [8]. For an introduction to directional tuning in the motor cortex refer to text books on neurophysiology. An in-depth coverage is found in [3].

# Data Sets

## Data Sets

All motor-cortex data used in this course are stored inside `data/motor_cortex/` in this repository. In Colab, run the setup cell below once; it clones the repository and defines `DATA_DIR`, `SPARSE_DATA_DIR`, `SELECTED_C1_DIR`, and `SELECTED_C3_DIR`.

The files are Matlab `.mat` files. In Python, a robust loading pattern is:

```python
from scipy.io import loadmat
mat = loadmat(path, squeeze_me=True, struct_as_record=False)
sparse_format = mat["SparseFormat"]
spike_matrices = sparse_format.Data
time_resolution_ms = sparse_format.TimeResolutionMS
cut_interval_ms = sparse_format.CutIntervalMS
```

`spike_matrices` is a length-6 NumPy array. Each entry corresponds to one movement direction and is stored as a sparse matrix with shape `(time_bins, n_trials)`. Convert a sparse matrix to a dense NumPy array with `.toarray()` when needed.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/schmitfe/Nawrot_CNS_Course.git"
REPO_NAME = "Nawrot_CNS_Course"


def prepare_repo() -> Path:
    if "google.colab" in sys.modules:
        repo_root = Path("/content") / REPO_NAME
        if not repo_root.exists():
            subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
        os.chdir(repo_root)
        return repo_root.resolve()
    return Path.cwd().resolve()


REPO_ROOT = prepare_repo()
DATA_DIR = REPO_ROOT / "data" / "motor_cortex"
SPARSE_DATA_DIR = DATA_DIR / "data_sparse_FromLinux"
SELECTED_C1_DIR = DATA_DIR / "SelectedDataC1"
SELECTED_C3_DIR = DATA_DIR / "SelectedDataC3"

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"DATA_DIR = {DATA_DIR}")


## 1. Visualization of neuronal population activity

The goal of this part is to visually represent the firing activity of a population of single neurons. The experiments involved acute recordings from one monkey. On each experimental day, at most 7 units were recorded [1]. Thus, in the data set provided for this module, only small subsets of neurons were recorded simultaneously. We therefore construct a so-called *pseudo-population*, i.e. we analyze neurons that were not recorded simultaneously but under identical task conditions. Start with files aligned to trial start (TS). These files contain `TS` in their filename and span `t = 0 ms` (trial start) to `t = 2000 ms`. The preparatory signal (PS) and reaction signal (RS) were presented at `t = 500 ms` and `t = 1500 ms`, respectively. Work in new Python code cells for this part.


**Tasks**
1. Perform the following analyses for a random set of `n = 10` neurons. Construct a list of file names first. Make sure that all computations below also work for other values of `n`: define a variable such as `n_neurons` instead of hard-coding the number 10. A convenient starting point is the `glob` module from Python’s standard library.


2. Construct an empty 3D matrix `S` with shape `(T, n, d)`, where `T = 2000` is the number of time bins at 1 ms resolution, `n` is the number of neurons, and `d = 6` is the number of movement directions.


3. Iterate across the selected file names. Keep the loop length fully data-driven so that your code still works when you change the file list.
  
  a. Load one data file.
  
  b. Loop across movement directions.
  
  c. For each direction, average the binary spike matrix across trials. This yields a 1D array of length `T = 2000`. Use explicit indexing so that you assign the result to the correct neuron and direction in `S`.


4. The 3d matrix $S$ now contains the trial-averaged activity of all neurons and all directions. Estimate the time-resolved firing rate using convolution with a kernel vector $k$.

5. Choose movement directions "left" and "right" (see Fig. 1 in [1]) and display the trial-averaged firing rates of all neurons in two color-coded panels. Define the time axis explicitly. Use the same color scale and axis limits in both panels. Mark PS and RS with vertical white or gray lines. Does the population activity differ between the two directions?


6. Perform PCA on the neuronal population activity. If `S` has shape `(T, n, d)`, one convenient approach is to transpose it to `(T, d, n)` and then reshape it to a 2D matrix of shape `(T * d, n)`. The PCA output should then have shape `(T * d, n_components)`. Plot the time evolution of the first 3 principal components using the same time axis as above. *Hint:* You can use `sklearn.decomposition.PCA`.


7. Choose the first two principal components, PC1 and PC2, and plot the trajectory in 2D PC space over time for each of the 6 directions. Optional: repeat this for the first three PCs and use a 3D plot.


8. Make sure your code is clean and well documented. Then repeat the analysis for different neuron sets and different population sizes. This should work simply by changing the file list or `n_neurons`, without rewriting the analysis code.


## 2. Decoding of movement direction from population activity

The idea is now to *decode* or *predict* movement direction from the activity of a neuronal population. Because the number of directions is discrete (`d = 6`), this is a classification problem: each single-trial population vector belongs to one of six classes.

We will first predict the executed movement from neuronal activity recorded during the preparatory period between PS and RS. Choose a time window `W = [650 ms, 1400 ms]` and use the activity in that window (spike count per trial) to train and test a classifier. A practical strategy for this pseudo-population setting is to repeatedly hold out one randomly drawn trial per direction for testing and use the remaining trials for training.

If time permits, perform trial-resolved decoding as well. In that case, use a sliding window of width `w` to count spikes over time. For a single test trial, this produces a time-resolved feature matrix with shape `(N, T)`.

Note that only small numbers of simultaneously recorded single units were available per experimental day. We therefore treat neurons as independently sampled and construct a pseudo-population [1]. Since single units were not recorded simultaneously, trial numbers do not match across neurons, and the number of available trials also differs across neurons and directions. A random sampling strategy is therefore appropriate. Define a set of `N` neurons (at least 10). For each neuron and direction, randomly draw one single trial for the test set and use the remaining trials to train a classifier such as *Linear Discriminant Analysis* or a *Naïve Bayes classifier*.

In the monkey experiment there were three task conditions. For a compact teaching example, start with one of the curated subsets in `SELECTED_C1_DIR` or `SELECTED_C3_DIR`. If you want to scale up the analysis, you can also work with the larger sparse dataset, which additionally contains condition C2. During the final discussion, compare your results across conditions and dataset choices.


## References

[1] Rickert J, Riehle A, Aertsen A, Rotter S, Nawrot MP (2009) Dynamic encoding of movement direction in motor cortical neurons. Journal of Neuroscience 29: 13870-13882

[2] Georgopoulos et al. (1982) On the relations between the direction of two-dimensional arm movements and cell discharge in primate motor cortex. J Neurosci 2(11): 1527-37

[3] Motor Cortex in Voluntary Movements: a distributed system for distributed functions (2005) Alexa Riehle, Eilon Vaadia (Eds.) CRC Press, Boca Raton, Florida, USA

[4] Mehring et al. (2003) Inference of hand movements from local field potentials in monkey motor cortex. Nat Neurosci 6(12): 1253-54

[5] Strube-Bloss MF, Nawrot MP, Menzel R (2011) Mushroom body output neurons encode odor-reward associations. The Journal of Neuroscience 31: 3129–3140.

[6] Nawrot M, Aertsen A, Rotter S (1999) Single-trial estimation of neuronal firing rates - From single neuron spike trains to population activity. J Neurosci Meth 94: 81-92

[7] Bastian, A., Schoner, G., and Riehle, A. (2003). Preshaping and continuous evolution of motor cortical representations during movement preparation. European Journal of Neuroscience, 18(7):2047–2058

[8] Nawrot MP (2010) Analysis and interpretation of interval and count variability in neural spikes trains. In: Grün S, Rotter S (eds) Analysis of parallel spikes trains. Springer Series in Computational Neuroscience 7. Springer Verlag, New York, Berlin, pp 34-58